In [1]:
import torch

### set up CUDA as device if available
if torch.cuda.is_available():
    print("GPU is available")
    device = torch.device("cuda")
    cuda_id = torch.cuda.current_device()
    print(f"ID of current CUDA device:{torch.cuda.current_device()}")
    print(f"Name of current CUDA device:{torch.cuda.get_device_name(cuda_id)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("GPU is not available, using CPU")
    device = torch.device("cpu")
print(f"device: {device}")

GPU is available
cuda
CUDA version: 12.4
ID of current CUDA device:0
Name of current CUDA device:NVIDIA H200


In [2]:
import numpy as np
import random
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm, trange
from IPython import display
from sklearn.metrics import average_precision_score
# from conf_and_plot import confusion_matrix_plots
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import KFold
from sklearn.model_selection import ParameterGrid
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from scipy.stats import kruskal
import torch.nn.functional as F
from IPython.display import clear_output
import statistics

def seed_all(seed):
    if not seed:
        seed = 10

    print("[ Using Seed : ", seed, " ]")

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

seed_all(2025)

[ Using Seed :  2025  ]


In [3]:
scint_thresh = 0.1 # set the phase scintillation threshold

### following variable not being used anywhere
scint_outlier_thresh = 5. # set the value that determines phase scintillation outliers (these data samples will be removed)

processed_data_2015 = pd.read_csv("processed_data_2015.csv")
processed_data_2015 = processed_data_2015.drop(processed_data_2015.columns[0], axis=1)
predicted_label = 'sigmaPhi projected to vertical at prediction time [radians]'
y = processed_data_2015[predicted_label].values
print(y.shape)

X_fSelect = processed_data_2015.drop(predicted_label, axis=1)
X_fSelect = X_fSelect.values
print(X_fSelect.shape)

(4465846,)
(4465846, 15)


In [4]:
class simple_timeSeries_CNN(nn.Module):
    def __init__(self, cnn_filters, fc_size, dropout_p, loss, sequence_length):
        super(simple_timeSeries_CNN, self).__init__()
        
        self.loss = loss
        self.sequence_length = sequence_length

        self.conv1 = nn.Conv1d(in_channels=15, out_channels=cnn_filters, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(num_features=cnn_filters)
        self.dropout = nn.Dropout(dropout_p)
        self.conv2 = nn.Conv1d(in_channels=cnn_filters, out_channels=cnn_filters*2, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(num_features=cnn_filters*2)
        self.pool = nn.MaxPool1d(2)
        
        ### for simple, non true time-series CNN:
        # num_features = 25, MaxPool1d_size=2
        # 12 = round(25 / 2)
        # 6 = round(12 / 2)
        # self.fc1 = nn.Linear(cnn_filters * 2 * 6, fc_size)
        
        ### for true time-series CNN:
        # sequence_length, MaxPool1d_size=2
        # round(sequence_length / 2) = sequence_length // 2
        # per maxpool layer = sequence_length // 4
        
        self.fc1 = nn.Linear(cnn_filters * 2 * (sequence_length // 4), fc_size)
        # self.fc2 = nn.Linear(fc_size, 1)
        self.fc2 = nn.Linear(fc_size, sequence_length)

    def forward(self, x):
        x = self.pool(torch.relu(self.bn1(self.conv1(x))))
        x = self.dropout(x)
        x = self.pool(torch.relu(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1) # Flatten the output for the fully connected layers
        x = torch.relu(self.fc1(x))
        
        # second fully connected layer + reshape to [batch_size, sequence_length, 1]
        x = self.fc2(x).view(x.size(0), self.sequence_length, 1)

        if self.loss == 'focal':
            # x = torch.sigmoid(self.fc2(x))
            x = torch.sigmoid(x)
        elif self.loss == 'bce':
            # x = self.fc2(x)
            pass
        
        return x

In [5]:
def true_skill_score(y_true, y_pred):
    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    tss = (TP / (TP + FN)) - (FP / (FP + TN))
    return tss

def heidke_skill_score(y_true, y_pred):
    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    
    numerator = 2 * (TP * TN - FP * FN)
    denominator = (TP + FP) * (FP + TN) + (TP + FN) * (TN + FP)
    
    # Avoid division by zero
    if denominator == 0:
        return 0.0
    
    hss = numerator / denominator
    return hss

In [6]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction  # 'mean', 'sum', or 'none'

    def forward(self, inputs, targets):
        BCE_loss = F.binary_cross_entropy(inputs, targets, reduction='none')
        p_t = inputs * targets + (1 - inputs) * (1 - targets)
        focal_loss = self.alpha * (1 - p_t) ** self.gamma * BCE_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        elif self.reduction == 'none':
            return focal_loss
        else:
            raise ValueError(f"Invalid reduction method: {self.reduction}")

In [7]:
def train_model_with_cv(model, train_dataloader, val_dataloader, criterion, optimizer, scheduler, num_epochs):

    epochs = []
    training_loss = []
    validation_loss = []
    training_tss = []
    validation_tss = []
    training_hss = []
    validation_hss = []

    for epoch in range(num_epochs):

        ### training loop
        model.train()
        running_loss = 0.0
        predicted_training_labels = np.array([])
        y_train = np.array([])

        # for batch_inputs, batch_labels in tqdm(train_dataloader):
        for batch_inputs, batch_labels in train_dataloader:
            
            batch_inputs = batch_inputs.to(device)
            batch_labels = batch_labels.to(device)
            
            optimizer.zero_grad()
            output = model(batch_inputs)
            
            output.to(device)
            
            loss = criterion(output, batch_labels.float())
            loss.backward()
            optimizer.step()
            
            if scheduler == False:
                pass
            else:
                scheduler.step()
            
            running_loss += loss.item()
            with torch.no_grad(): predicted_training_labels = np.append(predicted_training_labels, output.cpu())
            with torch.no_grad(): y_train = np.append(y_train, batch_labels.cpu())

        predicted_training_labels = np.where(predicted_training_labels > 0.1, 1, 0)

        train_loss = running_loss / len(train_dataloader)
        train_tss = true_skill_score(y_train.astype(int), predicted_training_labels.astype(int))
        train_hss = heidke_skill_score(y_train.astype(int), predicted_training_labels.astype(int))

        ### validation loop
        model.eval()
        with torch.no_grad():
            running_loss = 0.0
            predicted_val_labels = np.array([])
            y_val = np.array([])

            # for batch_inputs, labels in tqdm(val_dataloader):
            for batch_inputs, batch_labels in val_dataloader:
            
                batch_inputs = batch_inputs.to(device)
                batch_labels = batch_labels.to(device)
                
                outputs = model(batch_inputs)
            
                outputs.to(device)
                
                outputs = model(batch_inputs)
                loss = criterion(outputs, batch_labels.float())
                running_loss += loss.item()
                predicted_val_labels = np.append(predicted_val_labels, outputs.cpu())
                y_val = np.append(y_val, batch_labels.cpu())

        predicted_val_labels = np.where(predicted_val_labels > 0.1, 1, 0)

        val_loss = running_loss / len(val_dataloader)
        val_tss = true_skill_score(y_val.astype(int), predicted_val_labels.astype(int))
        val_hss = heidke_skill_score(y_val.astype(int), predicted_val_labels.astype(int))

        epochs.append(epoch)
        training_loss.append(train_loss)
        validation_loss.append(val_loss)
        training_tss.append(train_tss)
        validation_tss.append(val_tss)
        training_hss.append(train_hss)
        validation_hss.append(val_hss)

    return training_hss, validation_hss, training_tss, validation_tss


In [8]:
class CNNBinaryTimeSeriesClassifier():
    def __init__(self, cnn_filters=8, fc_size=64, sequence_length=60, dropout_p=0.1, batch_size=128, 
                 optimizer_type = 'adam', lr=0.001, sgd_momentum=0.9, weight_decay=0.0001,
                 loss='bce', bce_pos_class_weight=50, focal_alpha=0.25, focal_gamma=2,
                 num_epochs=5, cv_folds=5, scheduler_t=0):
        self.cnn_filters = cnn_filters
        self.fc_size = fc_size
        self.sequence_length = sequence_length
        self.dropout_p = dropout_p
        self.batch_size = batch_size
        self.optimizer_type = optimizer_type
        self.lr = lr        
        self.weight_decay = weight_decay
        self.sgd_momentum = sgd_momentum
        self.loss = loss
        self.bce_pos_class_weight = bce_pos_class_weight
        self.focal_alpha = focal_alpha
        self.focal_gamma = focal_gamma
        self.num_epochs = num_epochs
        self.cv_folds = cv_folds
        self.scheduler_t = scheduler_t
        
    def reshape_data_to_seq_length(self, data, seq_len):
        num_samples = data.shape[0]
        input_features = data.shape[1]
        num_batches = num_samples // seq_len
        data = data[:num_batches * seq_len] # truncate points that don't divide evenly
        reshaped_data = data.reshape(num_batches, seq_len, input_features)
        return reshaped_data

    def reshape_labels_to_seq_length(self, labels, seq_len):
        labels = torch.from_numpy(labels) if isinstance(labels, np.ndarray) else labels
        num_samples = labels.shape[0]
        num_batches = num_samples // seq_len
        labels = labels[:num_batches * seq_len] # truncate points that don't divide evenly
        labels = labels.view(num_batches, seq_len, -1)
        return labels
    
    def fit(self, X, y):

        cv_tss_values = []
        cv_hss_values = []

        tscv = TimeSeriesSplit(n_splits=self.cv_folds)
        for i, (train_index, val_index) in enumerate(tscv.split(X)):
            
            model = simple_timeSeries_CNN(cnn_filters=self.cnn_filters, 
                                          fc_size=self.fc_size, 
                                          dropout_p=self.dropout_p, 
                                          loss=self.loss, 
                                          sequence_length=self.sequence_length)
            model.to(device)
            
            if self.loss == 'bce':
                criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([self.bce_pos_class_weight]).to(device))
            elif self.loss == 'focal':
                criterion = FocalLoss(alpha=self.focal_alpha, gamma=self.focal_gamma)

            if self.optimizer_type == 'adam':
                optimizer = optim.Adam(model.parameters(), lr=self.lr, weight_decay=self.weight_decay)
            elif self.optimizer_type == 'sgd':
                optimizer = optim.SGD(model.parameters(), lr=self.lr, momentum=self.sgd_momentum, weight_decay=self.weight_decay)

            if self.scheduler_t > 0:
                scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.scheduler_t)
            else:
                scheduler = False

            X_train, X_val = X[train_index], X[val_index]
            y_train, y_val = y[train_index], y[val_index]
            
            scaler_X = RobustScaler()
            scaler_X = scaler_X.fit(X_train)

            X_train_scaled = scaler_X.transform(X_train)
            X_val_scaled = scaler_X.transform(X_val)
            
            X_train_scaled = self.reshape_data_to_seq_length(X_train_scaled, self.sequence_length)
            X_val_scaled = self.reshape_data_to_seq_length(X_val_scaled, self.sequence_length)

            train_labels = self.reshape_labels_to_seq_length(y_train, self.sequence_length)
            val_labels = self.reshape_labels_to_seq_length(y_val, self.sequence_length)
            
            train_inputs = torch.tensor(X_train_scaled, dtype=torch.float32).transpose(1, 2)  # Shape: [batch_size, 15, 1]
            train_inputs = train_inputs.to(device)
            train_labels = train_labels.to(device)
            train_dataset = TensorDataset(train_inputs, train_labels)
            train_dataloader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=False, worker_init_fn=seed_worker)

            val_inputs = torch.tensor(X_val_scaled, dtype=torch.float32).transpose(1, 2)  # Shape: [batch_size, 15, 1]
            val_inputs = val_inputs.to(device)
            val_labels = val_labels.to(device)
            val_dataset = TensorDataset(val_inputs, val_labels)
            val_dataloader = DataLoader(val_dataset, batch_size=self.batch_size, shuffle=False, worker_init_fn=seed_worker)
            
            training_hss, validation_hss, training_tss, validation_tss = \
            train_model_with_cv(model, 
                 train_dataloader, val_dataloader, 
                 criterion, optimizer, scheduler,
                 num_epochs=self.num_epochs)
            cv_tss_values.append(validation_tss[-1])
            cv_hss_values.append(validation_hss[-1])

            if i == 0:
                total_params = sum(p.numel() for p in model.parameters())
                print(f'Total number of parameters: {total_params}')
                
            print(f"Fold {i}: train tss = {training_tss[-1]:.4f} : val tss = {validation_tss[-1]:.4f} ::: train hss = {training_hss[-1]:.4f} : val hss = {validation_hss[-1]:.4f}")
        
        return cv_hss_values, cv_tss_values
    

In [9]:
training_data_size = 50000
training_data_size = 500000

X_train, X_test, \
    y_train, y_test, \
        idx_train, idx_test = train_test_split(X_fSelect, y, range(len(y)), train_size=training_data_size, shuffle=False)

# ### small test
# param_grid = {
#     'cnn_filters': [256],   
#     'fc_size': [256, 1048],  
#     'sequence_length': [100],   
#     'dropout_p': [0.6],  
#     'batch_size': [64],  
#     'lr': [0.03],  
#     'optimizer_type': ['sgd'],  
#     'weight_decay':[0.05],  
#     'loss': ['bce'],
#     'bce_pos_class_weight': [35],  
#     'num_epochs': [20],  
#     'cv_folds': [5], 
# }

### iteration 1
param_grid = {
    'cnn_filters': [16, 64],   
    'fc_size': [256, 1048],  
    'sequence_length': [15, 45],   
    'dropout_p': [0.1, 0.5],  
    'batch_size': [64, 512],  
    'lr': [0.0001, 0.01],  
    'optimizer_type': ['sgd','adam'],  
    'weight_decay':[0.001, 0.01],  
    'loss': ['bce'],
    'bce_pos_class_weight': [35, 75],  
    'num_epochs': [40],  
    'cv_folds': [5], 
}

### iteration 2
param_grid = {
    'cnn_filters': [16, 64],   
    'fc_size': [256, 1048],  
    'sequence_length': [30, 45],  ### changed
    'dropout_p': [0.1, 0.5],  
    'batch_size': [512, 1024],  ### changed
    'lr': [0.001, 0.01],  ### changed
    'optimizer_type': ['sgd','adam'],  
    'weight_decay':[0.001, 0.01],  
    'loss': ['bce'],
    'bce_pos_class_weight': [35, 65],  ### changed
    'num_epochs': [40],  
    'cv_folds': [5], 
}

### iteration 3
param_grid = {
    'cnn_filters': [32, 128],   ### changed
    'fc_size': [512, 2048],  ### changed
    'sequence_length': [40, 45],  ### changed
    'dropout_p': [0.1, 0.5],  
    'batch_size': [512, 1024],  
    'lr': [0.005, 0.025],  ### changed
    'optimizer_type': ['sgd','adam'],  
    'weight_decay':[0.001, 0.025],  ### changed
    'loss': ['bce'],
    'bce_pos_class_weight': [25, 50],  ### changed
    'num_epochs': [40],  
    'cv_folds': [5], 
}

### iteration 4
param_grid = {
    'cnn_filters': [64, 256],  ### changed
    'fc_size': [512, 2048],  
    'sequence_length': [40, 45],  
    'dropout_p': [0.1, 0.5],  
    'batch_size': [512, 1024],  
    'lr': [0.005, 0.05],  ### changed
    'optimizer_type': ['sgd','adam'],  
    'weight_decay':[0.01, 0.05],  ### changed
    'loss': ['bce'],
    'bce_pos_class_weight': [40, 75],  ### changed
    'num_epochs': [40],  
    'cv_folds': [5], 
}

### iteration 5
param_grid = {
    'cnn_filters': [192, 512],  ### changed
    'fc_size': [1536, 2560],  ### changed
    'sequence_length': [40, 45],  
    'dropout_p': [0.1, 0.25],  ### changed
    'batch_size': [512, 1024],  
    'lr': [0.005, 0.05],  
    'optimizer_type': ['sgd','adam'],  
    'weight_decay':[0.01, 0.05],  
    'loss': ['bce'],
    'bce_pos_class_weight': [15, 60],  ### changed
    'num_epochs': [40],  
    'cv_folds': [5], 
}

### iteration 6
param_grid = {
    'cnn_filters': [320, 512],  ### changed
    'fc_size': [768, 1280],  ### changed
    'sequence_length': [40, 45],  
    'dropout_p': [0.2, 0.4],  ### changed
    'batch_size': [512, 1024],  
    'lr': [0.0025, 0.075],  ### changed
    'optimizer_type': ['sgd','adam'],  
    'weight_decay':[0.0075, 0.075],  ### changed
    'loss': ['bce'],
    'bce_pos_class_weight': [15, 60],  
    'num_epochs': [40],  
    'cv_folds': [5], 
}

### iteration 7
param_grid = {
    'cnn_filters': [32, 512],  ### changed
    'fc_size': [1536, 2048],  ### changed
    'sequence_length': [35, 45],  ### changed
    'dropout_p': [0.1, 0.25],  ### changed
    'batch_size': [512, 1024],  
    'lr': [0.005, 0.01],  ### changed
    'optimizer_type': ['sgd','adam'],  
    'weight_decay':[0.01, 0.05],  ### changed
    'loss': ['bce'],
    'bce_pos_class_weight': [10, 70],  ### changed
    'num_epochs': [40],  
    'cv_folds': [5], 
}

### iteration 8
param_grid = {
    'cnn_filters': [96, 256],  ### changed
    'fc_size': [1536, 2048],  
    'sequence_length': [10, 45],  ### changed
    'dropout_p': [0.1, 0.25],  
    'batch_size': [512, 1024],  
    'lr': [0.005, 0.01],  
    'optimizer_type': ['sgd','adam'],  
    'weight_decay':[0.02, 0.04],  ### changed
    'loss': ['bce'],
    'bce_pos_class_weight': [5, 80],  ### changed
    'num_epochs': [40],  
    'cv_folds': [5], 
}

### iteration 9
param_grid = {
    'cnn_filters': [32, 128],  ### changed
    'fc_size': [1024, 2048],  ### changed
    'sequence_length': [45],  ### changed, removed 1 dimension
    'dropout_p': [0.1, 0.25],  
    'batch_size': [512, 1024],  
    'lr': [0.005, 0.025],  ### changed
    'optimizer_type': ['sgd','adam'],  
    'weight_decay':[0.02, 0.04],  
    'loss': ['bce'],
    'bce_pos_class_weight': [10, 65],  ### changed
    'num_epochs': [40],  
    'cv_folds': [5], 
}

### iteration 10
param_grid = {
    'cnn_filters': [48, 256],  ### changed
    'fc_size': [1024, 2048],  
    'sequence_length': [45],  
    'dropout_p': [0.05, 0.35],  ### changed
    'batch_size': [512, 1024],  
    'lr': [0.005, 0.01],  ### changed
    'optimizer_type': ['sgd','adam'],  
    'weight_decay':[0.03, 0.05],  ### changed
    'loss': ['bce'],
    'bce_pos_class_weight': [35, 50],  ### changed
    'num_epochs': [40],  
    'cv_folds': [5], 
}

param_combinations = list(ParameterGrid(param_grid))

grid = pd.DataFrame()
tss_distributions = []
hss_distributions = []
mean_tss_scores = []
mean_hss_scores = []
for params in param_combinations:

    print(str(param_combinations.index(params)) + "/" + str(len(param_combinations)))
    print(params)

    cnn_time_series_classifier = CNNBinaryTimeSeriesClassifier(**params)
    cv_hss_values, cv_tss_values = cnn_time_series_classifier.fit(X_train, y_train)
    
    tss_distributions.append(cv_tss_values)
    hss_distributions.append(cv_hss_values)
    mean_cv_tss = sum(cv_tss_values)/len(cv_tss_values)
    mean_cv_hss = sum(cv_hss_values)/len(cv_hss_values)
    print(f"mean cv TSS: {mean_cv_tss:.4f}")
    print(f"mean cv HSS: {mean_cv_hss:.4f}")
    mean_tss_scores.append(mean_cv_tss)
    mean_hss_scores.append(mean_cv_hss)
    
    if params == param_combinations[0]:
        grid = pd.DataFrame(params, index=[0])
    else:
        grid = pd.concat([grid, pd.DataFrame([params])], ignore_index=True)

grid['tss'] = [round(num, 4) for num in mean_tss_scores]
grid['hss'] = [round(num, 4) for num in mean_hss_scores]
grid = grid.sort_values(by=['tss', 'hss'], ascending=False)
grid = grid.drop(columns=[col for col in ['num_epochs','cv_folds','loss','scheduler_t'] if col in grid.columns], axis=1)
grid.head(20)


0/256
{'batch_size': 512, 'bce_pos_class_weight': 35, 'cnn_filters': 48, 'cv_folds': 5, 'dropout_p': 0.05, 'fc_size': 1024, 'loss': 'bce', 'lr': 0.005, 'num_epochs': 40, 'optimizer_type': 'sgd', 'sequence_length': 45, 'weight_decay': 0.03}
Total number of parameters: 1144909
Fold 0: train tss = 0.6990 : val tss = 0.6736 ::: train hss = 0.2382 : val hss = 0.2715
Fold 1: train tss = 0.7045 : val tss = 0.6725 ::: train hss = 0.2405 : val hss = 0.2144
Fold 2: train tss = 0.6781 : val tss = 0.6260 ::: train hss = 0.1926 : val hss = 0.0753
Fold 3: train tss = 0.6647 : val tss = 0.5509 ::: train hss = 0.1756 : val hss = 0.1048
Fold 4: train tss = 0.7102 : val tss = 0.6360 ::: train hss = 0.2250 : val hss = 0.1864
mean cv TSS: 0.6318
mean cv HSS: 0.1705
1/256
{'batch_size': 512, 'bce_pos_class_weight': 35, 'cnn_filters': 48, 'cv_folds': 5, 'dropout_p': 0.05, 'fc_size': 1024, 'loss': 'bce', 'lr': 0.005, 'num_epochs': 40, 'optimizer_type': 'sgd', 'sequence_length': 45, 'weight_decay': 0.05}
Tota

Total number of parameters: 2273357
Fold 0: train tss = 0.6927 : val tss = 0.6219 ::: train hss = 0.2376 : val hss = 0.1955
Fold 1: train tss = 0.6411 : val tss = 0.7150 ::: train hss = 0.2012 : val hss = 0.1486
Fold 2: train tss = 0.6161 : val tss = 0.6235 ::: train hss = 0.1515 : val hss = 0.0930
Fold 3: train tss = 0.6303 : val tss = 0.6591 ::: train hss = 0.1628 : val hss = 0.2194
Fold 4: train tss = 0.6103 : val tss = 0.6711 ::: train hss = 0.1505 : val hss = 0.1528
mean cv TSS: 0.6581
mean cv HSS: 0.1619
12/256
{'batch_size': 512, 'bce_pos_class_weight': 35, 'cnn_filters': 48, 'cv_folds': 5, 'dropout_p': 0.05, 'fc_size': 2048, 'loss': 'bce', 'lr': 0.01, 'num_epochs': 40, 'optimizer_type': 'sgd', 'sequence_length': 45, 'weight_decay': 0.03}
Total number of parameters: 2273357
Fold 0: train tss = 0.7695 : val tss = 0.6760 ::: train hss = 0.3083 : val hss = 0.2794
Fold 1: train tss = 0.7173 : val tss = 0.6757 ::: train hss = 0.2506 : val hss = 0.1338
Fold 2: train tss = 0.6919 : val

Total number of parameters: 1144909
Fold 0: train tss = 0.7239 : val tss = 0.6676 ::: train hss = 0.2710 : val hss = 0.3055
Fold 1: train tss = 0.6181 : val tss = 0.7582 ::: train hss = 0.1736 : val hss = 0.1545
Fold 2: train tss = 0.6172 : val tss = 0.6551 ::: train hss = 0.1505 : val hss = 0.1241
Fold 3: train tss = 0.6251 : val tss = 0.6381 ::: train hss = 0.1547 : val hss = 0.2186
Fold 4: train tss = 0.6106 : val tss = 0.6517 ::: train hss = 0.1495 : val hss = 0.1415
mean cv TSS: 0.6742
mean cv HSS: 0.1888
23/256
{'batch_size': 512, 'bce_pos_class_weight': 35, 'cnn_filters': 48, 'cv_folds': 5, 'dropout_p': 0.35, 'fc_size': 1024, 'loss': 'bce', 'lr': 0.01, 'num_epochs': 40, 'optimizer_type': 'adam', 'sequence_length': 45, 'weight_decay': 0.05}
Total number of parameters: 1144909
Fold 0: train tss = 0.6945 : val tss = 0.6101 ::: train hss = 0.2521 : val hss = 0.2185
Fold 1: train tss = 0.6069 : val tss = 0.6647 ::: train hss = 0.1598 : val hss = 0.2021
Fold 2: train tss = 0.6042 : va

Total number of parameters: 6221357
Fold 0: train tss = 0.7496 : val tss = 0.6591 ::: train hss = 0.2933 : val hss = 0.2758
Fold 1: train tss = 0.7208 : val tss = 0.7191 ::: train hss = 0.2606 : val hss = 0.0982
Fold 2: train tss = 0.7245 : val tss = 0.6336 ::: train hss = 0.2611 : val hss = 0.1012
Fold 3: train tss = 0.7124 : val tss = 0.6504 ::: train hss = 0.2173 : val hss = 0.2804
Fold 4: train tss = 0.7070 : val tss = 0.6255 ::: train hss = 0.2220 : val hss = 0.2059
mean cv TSS: 0.6576
mean cv HSS: 0.1923
34/256
{'batch_size': 512, 'bce_pos_class_weight': 35, 'cnn_filters': 256, 'cv_folds': 5, 'dropout_p': 0.05, 'fc_size': 1024, 'loss': 'bce', 'lr': 0.005, 'num_epochs': 40, 'optimizer_type': 'adam', 'sequence_length': 45, 'weight_decay': 0.03}
Total number of parameters: 6221357
Fold 0: train tss = 0.7510 : val tss = 0.6024 ::: train hss = 0.2990 : val hss = 0.3466
Fold 1: train tss = 0.6228 : val tss = 0.7525 ::: train hss = 0.1763 : val hss = 0.1324
Fold 2: train tss = 0.5885 : 

Total number of parameters: 12035629
Fold 0: train tss = 0.8210 : val tss = 0.6328 ::: train hss = 0.4050 : val hss = 0.2321
Fold 1: train tss = 0.7699 : val tss = 0.6742 ::: train hss = 0.3159 : val hss = 0.2691
Fold 2: train tss = 0.7695 : val tss = 0.6652 ::: train hss = 0.3013 : val hss = 0.1698
Fold 3: train tss = 0.7018 : val tss = 0.6477 ::: train hss = 0.2138 : val hss = 0.2516
Fold 4: train tss = 0.7425 : val tss = 0.5913 ::: train hss = 0.2529 : val hss = 0.2324
mean cv TSS: 0.6422
mean cv HSS: 0.2310
45/256
{'batch_size': 512, 'bce_pos_class_weight': 35, 'cnn_filters': 256, 'cv_folds': 5, 'dropout_p': 0.05, 'fc_size': 2048, 'loss': 'bce', 'lr': 0.01, 'num_epochs': 40, 'optimizer_type': 'sgd', 'sequence_length': 45, 'weight_decay': 0.05}
Total number of parameters: 12035629
Fold 0: train tss = 0.7369 : val tss = 0.6439 ::: train hss = 0.2891 : val hss = 0.2518
Fold 1: train tss = 0.7363 : val tss = 0.6509 ::: train hss = 0.2714 : val hss = 0.2223
Fold 2: train tss = 0.6864 : 

...

Total number of parameters: 2273357
Fold 0: train tss = 0.6217 : val tss = 0.6726 ::: train hss = 0.1677 : val hss = 0.2472
Fold 1: train tss = 0.6719 : val tss = 0.7615 ::: train hss = 0.1998 : val hss = 0.1739
Fold 2: train tss = 0.6824 : val tss = 0.6718 ::: train hss = 0.1847 : val hss = 0.1184
Fold 3: train tss = 0.6899 : val tss = 0.5968 ::: train hss = 0.1680 : val hss = 0.1336
Fold 4: train tss = 0.6630 : val tss = 0.6928 ::: train hss = 0.1598 : val hss = 0.1326
mean cv TSS: 0.6791
mean cv HSS: 0.1611
221/256
{'batch_size': 1024, 'bce_pos_class_weight': 50, 'cnn_filters': 48, 'cv_folds': 5, 'dropout_p': 0.35, 'fc_size': 2048, 'loss': 'bce', 'lr': 0.01, 'num_epochs': 40, 'optimizer_type': 'sgd', 'sequence_length': 45, 'weight_decay': 0.05}
Total number of parameters: 2273357
Fold 0: train tss = 0.6006 : val tss = 0.6805 ::: train hss = 0.1520 : val hss = 0.2542
Fold 1: train tss = 0.6609 : val tss = 0.7261 ::: train hss = 0.1883 : val hss = 0.1814
Fold 2: train tss = 0.6662 : v

Total number of parameters: 6221357
Fold 0: train tss = 0.7157 : val tss = 0.5978 ::: train hss = 0.2671 : val hss = 0.1696
Fold 1: train tss = 0.6569 : val tss = 0.7212 ::: train hss = 0.1887 : val hss = 0.1688
Fold 2: train tss = 0.6240 : val tss = 0.6425 ::: train hss = 0.1374 : val hss = 0.0969
Fold 3: train tss = 0.6227 : val tss = 0.6679 ::: train hss = 0.1296 : val hss = 0.1947
Fold 4: train tss = 0.6275 : val tss = 0.6473 ::: train hss = 0.1426 : val hss = 0.1294
mean cv TSS: 0.6553
mean cv HSS: 0.1519
232/256
{'batch_size': 1024, 'bce_pos_class_weight': 50, 'cnn_filters': 256, 'cv_folds': 5, 'dropout_p': 0.05, 'fc_size': 2048, 'loss': 'bce', 'lr': 0.005, 'num_epochs': 40, 'optimizer_type': 'sgd', 'sequence_length': 45, 'weight_decay': 0.03}
Total number of parameters: 12035629
Fold 0: train tss = 0.6792 : val tss = 0.6739 ::: train hss = 0.2080 : val hss = 0.2734
Fold 1: train tss = 0.7113 : val tss = 0.7431 ::: train hss = 0.2318 : val hss = 0.1816
Fold 2: train tss = 0.7545 

Total number of parameters: 6221357
Fold 0: train tss = 0.7784 : val tss = 0.6634 ::: train hss = 0.3074 : val hss = 0.2402
Fold 1: train tss = 0.6745 : val tss = 0.7258 ::: train hss = 0.2066 : val hss = 0.1629
Fold 2: train tss = 0.6541 : val tss = 0.6301 ::: train hss = 0.1600 : val hss = 0.0814
Fold 3: train tss = 0.6275 : val tss = 0.6221 ::: train hss = 0.1315 : val hss = 0.1471
Fold 4: train tss = 0.6217 : val tss = 0.6687 ::: train hss = 0.1362 : val hss = 0.1284
mean cv TSS: 0.6620
mean cv HSS: 0.1520
243/256
{'batch_size': 1024, 'bce_pos_class_weight': 50, 'cnn_filters': 256, 'cv_folds': 5, 'dropout_p': 0.35, 'fc_size': 1024, 'loss': 'bce', 'lr': 0.005, 'num_epochs': 40, 'optimizer_type': 'adam', 'sequence_length': 45, 'weight_decay': 0.05}
Total number of parameters: 6221357
Fold 0: train tss = 0.6769 : val tss = 0.6491 ::: train hss = 0.1903 : val hss = 0.1976
Fold 1: train tss = 0.6544 : val tss = 0.7476 ::: train hss = 0.1935 : val hss = 0.1290
Fold 2: train tss = 0.5850 

Total number of parameters: 12035629
Fold 0: train tss = 0.6664 : val tss = 0.6709 ::: train hss = 0.1932 : val hss = 0.2560
Fold 1: train tss = 0.7020 : val tss = 0.7547 ::: train hss = 0.2239 : val hss = 0.1427
Fold 2: train tss = 0.6447 : val tss = 0.6098 ::: train hss = 0.1519 : val hss = 0.0692
Fold 3: train tss = 0.6398 : val tss = 0.5556 ::: train hss = 0.1317 : val hss = 0.1067
Fold 4: train tss = 0.6887 : val tss = 0.6345 ::: train hss = 0.1762 : val hss = 0.1492
mean cv TSS: 0.6451
mean cv HSS: 0.1448
254/256
{'batch_size': 1024, 'bce_pos_class_weight': 50, 'cnn_filters': 256, 'cv_folds': 5, 'dropout_p': 0.35, 'fc_size': 2048, 'loss': 'bce', 'lr': 0.01, 'num_epochs': 40, 'optimizer_type': 'adam', 'sequence_length': 45, 'weight_decay': 0.03}
Total number of parameters: 12035629
Fold 0: train tss = 0.5279 : val tss = 0.4438 ::: train hss = 0.1110 : val hss = 0.0817
Fold 1: train tss = 0.6462 : val tss = 0.7484 ::: train hss = 0.1829 : val hss = 0.1523
Fold 2: train tss = 0.6646

,batch_size,bce_pos_class_weight,cnn_filters,dropout_p,fc_size,lr,optimizer_type,sequence_length,weight_decay,tss,hss
165,1024,35,256,0.05,1024,0.010,sgd,45,0.05,0.6828,0.1942
57,512,35,256,0.35,2048,0.005,sgd,45,0.05,0.6827,0.1843
224,1024,50,256,0.05,1024,0.005,sgd,45,0.03,0.6826,0.1746
205,1024,50,48,0.05,2048,0.010,sgd,45,0.05,0.6800,0.1726
105,512,50,256,0.05,2048,0.005,sgd,45,0.05,0.6797,0.1773
220,1024,50,48,0.35,2048,0.010,sgd,45,0.03,0.6791,0.1611
97,512,50,256,0.05,1024,0.005,sgd,45,0.05,0.6774,0.1748
28,512,35,48,0.35,2048,0.010,sgd,45,0.03,0.6753,0.1967
49,512,35,256,0.35,1024,0.005,sgd,45,0.05,0.6747,0.1827
22,512,35,48,0.35,1024,0.010,adam,45,0.03,0.6742,0.1888


In [10]:
grid.sort_values(by=['hss', 'tss'], ascending=False).head(20)

,batch_size,bce_pos_class_weight,cnn_filters,dropout_p,fc_size,lr,optimizer_type,sequence_length,weight_decay,tss,hss
142,1024,35,48,0.05,2048,0.010,adam,45,0.03,0.6119,0.2621
143,1024,35,48,0.05,2048,0.010,adam,45,0.05,0.6492,0.2425
141,1024,35,48,0.05,2048,0.010,sgd,45,0.05,0.6637,0.2420
159,1024,35,48,0.35,2048,0.010,adam,45,0.05,0.6135,0.2395
169,1024,35,256,0.05,2048,0.005,sgd,45,0.05,0.6488,0.2384
163,1024,35,256,0.05,1024,0.005,adam,45,0.05,0.6208,0.2365
135,1024,35,48,0.05,1024,0.010,adam,45,0.05,0.6075,0.2354
8,512,35,48,0.05,2048,0.005,sgd,45,0.03,0.6254,0.2328
183,1024,35,256,0.35,1024,0.010,adam,45,0.05,0.6455,0.2321
44,512,35,256,0.05,2048,0.010,sgd,45,0.03,0.6422,0.2310


In [11]:
if grid.empty:
    print("grid empty")
else:
    grid.to_csv("1_big_cnn_ts_hypertune_grid_output_10.csv")
    print("grid output successful, saved as 1_big_cnn_ts_hypertune_grid_output_10.csv")
    

grid output successful, saved as 1_big_cnn_ts_hypertune_grid_output_10.csv
